# 06 — Combined Clinical + All-Images Merged PCA Clustering

Combines **all 5 image types merged PCA features** (550 components) with **clinical variables**
(Age, M/F, Educ, SES, MMSE, eTIV, nWBV, ASF) into one feature matrix and clusters
using Generalised Gower distance via `SampleDistClustering`.

CDR is **excluded** from the feature matrix and used only for post-hoc evaluation.

## Step 1 — Install dependencies

In [11]:
import subprocess, sys

def run_pip(*args):
    cmd = [sys.executable, "-m", "pip", "install", "--quiet"] + list(args)
    r   = subprocess.run(cmd, capture_output=True, text=True)
    tag = args[-1]
    if r.returncode == 0:
        print(f"  OK    {tag}")
    else:
        print(f"  FAIL  {tag}")
        print(r.stderr[-600:] if r.stderr else "(no stderr)")

run_pip("--no-deps", "db-robust-clust==0.1.10")
for pkg in ["kmedoids", "robust-mixed-dist", "openpyxl"]:
    run_pip(pkg)
print("Done.")

  OK    db-robust-clust==0.1.10
  OK    kmedoids
  OK    robust-mixed-dist
  OK    openpyxl
Done.


## Step 2 — Imports

In [12]:
import os
import numpy as np
import pandas as pd

import kmedoids
from db_robust_clust.models import SampleDistClustering
from robust_mixed_dist.mixed import generalized_gower_dist_matrix
from sklearn.metrics import (
    silhouette_score,
    adjusted_rand_score,
    normalized_mutual_info_score,
)

print("All imports successful.")

All imports successful.


## Step 3 — Load and merge data

Load the all-images-merged PCA features and the clinical table, then merge on `patient_id`.
CDR is saved separately for evaluation and is **not** included in the feature matrix.

In [ ]:
DATA_DIR = r"C:\Users\aregk\OneDrive\Desktop\Thesis_downoading_the_data"
NB_DIR   = os.path.join(DATA_DIR, "notebooks")

# ── PCA features (all 5 image types merged) ───────────────────────────────────
df_pca = pd.read_csv(os.path.join(NB_DIR, "pca_all_images_merged.csv"))
print(f"PCA file          : pca_all_images_merged.csv  {df_pca.shape}")
print(f"PCA ID sample     : {df_pca['patient_id'].head(3).tolist()}")

# ── Clinical data ─────────────────────────────────────────────────────────────
df_clin = pd.read_excel(
    os.path.join(NB_DIR, "oasis_cross-sectional-5708aa0a98d82080.xlsx")
)
if "ID" in df_clin.columns:
    df_clin = df_clin.rename(columns={"ID": "patient_id"})

df_clin["patient_id"] = df_clin["patient_id"].str.replace(r"_MR\d+$", "", regex=True)
df_clin = (
    df_clin
    .sort_values("CDR", na_position="last")
    .drop_duplicates(subset="patient_id", keep="first")
    .reset_index(drop=True)
)
print(f"\nClinical table    : {df_clin.shape}")

cdr_lookup = df_clin.set_index("patient_id")["CDR"]

KEEP_CLINICAL = ["patient_id", "Age", "M/F", "Educ", "SES", "MMSE", "eTIV", "nWBV", "ASF"]
df_merged = df_pca.merge(df_clin[KEEP_CLINICAL], on="patient_id", how="inner")
print(f"\nMerged (before preprocessing) : {df_merged.shape}")

## Step 4 — Preprocessing and feature matrix construction

Generalised Gower requires columns in a strict order: **quantitative → binary → multiclass**.

| Group | Columns | Count |
|-------|---------|-------|
| Quantitative (`p1`) | 550 PCA + Age + MMSE + eTIV + nWBV + ASF | 555 |
| Binary (`p2`) | M/F encoded as 1/0 | 1 |
| Multiclass (`p3`) | Educ + SES | 2 |

Missing values in Educ, SES, MMSE are filled with the column median.

In [14]:
df = df_merged.copy()

# ── Fill missing values ───────────────────────────────────────────────────────
for col in ["Educ", "SES", "MMSE"]:
    n_missing = df[col].isna().sum()
    if n_missing > 0:
        median_val = df[col].median()
        df[col] = df[col].fillna(median_val)
        print(f"  Filled {n_missing} missing in {col} with median={median_val}")

# ── Encode M/F as binary (M=1, F=0) ──────────────────────────────────────────
df["MF"] = (df["M/F"] == "M").astype(int)
df = df.drop(columns=["M/F"])

# ── Convert Educ and SES to integer (multiclass categories) ──────────────────
df["Educ"] = df["Educ"].astype(int)
df["SES"]  = df["SES"].astype(int)

# ── Build feature matrix in required column order ─────────────────────────────
pca_cols       = [c for c in df.columns if c not in
                  ["patient_id", "Age", "MF", "Educ", "SES", "MMSE", "eTIV", "nWBV", "ASF"]]
quant_clinical = ["Age", "MMSE", "eTIV", "nWBV", "ASF"]
binary_cols    = ["MF"]
multi_cols     = ["Educ", "SES"]

feature_cols = pca_cols + quant_clinical + binary_cols + multi_cols
patient_ids  = df["patient_id"].values
X            = df[feature_cols].to_numpy(dtype=np.float64)

P1 = len(pca_cols) + len(quant_clinical)   # quantitative
P2 = len(binary_cols)                       # binary
P3 = len(multi_cols)                        # multiclass

print(f"\nFeature matrix X  : {X.shape}")
print(f"  p1 (quantitative) = {P1}  ({len(pca_cols)} PCA + {len(quant_clinical)} clinical)")
print(f"  p2 (binary)       = {P2}  ({binary_cols})")
print(f"  p3 (multiclass)   = {P3}  ({multi_cols})")
print(f"\nAll feature columns ({len(feature_cols)}):")
print(f"  PCA     : {pca_cols[:3]} ... {pca_cols[-3:]}")
print(f"  Quant   : {quant_clinical}")
print(f"  Binary  : {binary_cols}")
print(f"  Multi   : {multi_cols}")

# ── Align CDR to the merged patient order ─────────────────────────────────────
cdr_series = cdr_lookup.reindex(patient_ids)
n_with_cdr = int(cdr_series.notnull().sum())
print(f"\nPatients with CDR : {n_with_cdr} / {len(patient_ids)}")

  Filled 181 missing in Educ with median=3.0
  Filled 200 missing in SES with median=2.0
  Filled 181 missing in MMSE with median=29.0

Feature matrix X  : (416, 558)
  p1 (quantitative) = 555  (550 PCA + 5 clinical)
  p2 (binary)       = 1  (['MF'])
  p3 (multiclass)   = 2  (['Educ', 'SES'])

All feature columns (558):
  PCA     : ['cor_PC1', 'cor_PC2', 'cor_PC3'] ... ['sbj_sag_PC108', 'sbj_sag_PC109', 'sbj_sag_PC110']
  Quant   : ['Age', 'MMSE', 'eTIV', 'nWBV', 'ASF']
  Binary  : ['MF']
  Multi   : ['Educ', 'SES']

Patients with CDR : 235 / 416


## Step 5 — Clustering with Generalised Gower distance

Run `SampleDistClustering` with `metric='ggower'` for both **k=3** and **k=4**.
The package computes the Generalised Gower distance matrix internally from `X`.

In [15]:
def fit_ggower(k):
    base = kmedoids.KMedoids(
        n_clusters=k, metric="precomputed", method="pam", random_state=42,
    )
    model = SampleDistClustering(
        clustering_method=base,
        metric="ggower",
        frac_sample_size=1.0,
        random_state=42,
        p1=P1, p2=P2, p3=P3,
        d1="euclidean",  # changed from robust_mahalanobis
        d2="sokal",
        d3="hamming",
    )
    print(f"  Fitting k={k} ...", end=" ", flush=True)
    model.fit(X)
    labels  = np.array(model.labels_, dtype=int)
    inertia = getattr(model, "inertia_",
              getattr(model.clustering_method, "inertia_", None))
    sizes   = {c: int((labels == c).sum()) for c in sorted(set(labels))}
    print(f"done  |  inertia={inertia:.4f}  |  sizes={sizes}")
    return labels

print("Clustering with Generalised Gower (d1=euclidean):")
labels_k3 = fit_ggower(k=3)
labels_k4 = fit_ggower(k=4)

Clustering with Generalised Gower (d1=euclidean):
  Fitting k=3 ... done  |  inertia=461.8922  |  sizes={np.int64(0): 78, np.int64(1): 183, np.int64(2): 155}
  Fitting k=4 ... done  |  inertia=420.7574  |  sizes={np.int64(0): 68, np.int64(1): 99, np.int64(2): 94, np.int64(3): 155}


## Step 6 — Compute Generalised Gower distance matrix

Needed explicitly for the Silhouette score.  This is the same distance that
`SampleDistClustering` computed internally during fitting.

In [16]:
print("Computing Generalised Gower distance matrix (d1=euclidean) ...")
D = generalized_gower_dist_matrix(
    X,
    p1=P1, p2=P2, p3=P3,
    d1="euclidean",
    d2="sokal",
    d3="hamming",
)
print(f"Distance matrix   : {D.shape}")
print(f"Value range       : [{D.min():.4f}, {D.max():.4f}]")

Computing Generalised Gower distance matrix (d1=euclidean) ...
Distance matrix   : (416, 416)
Value range       : [0.0000, 5.9782]


## Evaluation

All metrics computed for both k=3 and k=4.

**CDR → integer categories for ARI/NMI**: multiply CDR by 2 to remove the 0.5 decimal.

| CDR | × 2 | Meaning |
|-----|-----|---------|
| 0.0 | 0 | Healthy |
| 0.5 | 1 | Very mild / MCI |
| 1.0 | 2 | Mild dementia |
| 2.0 | 4 | Moderate dementia |

### Step 7 — Crosstab: cluster vs CDR

In [17]:
def show_crosstab(labels, k):
    df_res = pd.DataFrame({"patient_id": patient_ids, "cluster": labels})
    df_res = df_res.merge(
        cdr_series.reset_index().rename(columns={"CDR": "CDR"}),
        on="patient_id", how="left"
    )
    has_cdr = df_res[df_res["CDR"].notnull()].copy()
    has_cdr["CDR"] = has_cdr["CDR"].astype(str)

    print(f"\n{'═'*55}")
    print(f"  k={k}  —  {len(has_cdr)} patients with CDR")
    print(f"{'═'*55}")
    ct = pd.crosstab(has_cdr["cluster"], has_cdr["CDR"],
                     margins=True, margins_name="Total")
    print(ct.to_string())
    print("\nRow % (CDR proportion within each cluster):")
    pct = pd.crosstab(has_cdr["cluster"], has_cdr["CDR"],
                      normalize="index").mul(100).round(1)
    print(pct.to_string())

show_crosstab(labels_k3, k=3)
show_crosstab(labels_k4, k=4)


═══════════════════════════════════════════════════════
  k=3  —  235 patients with CDR
═══════════════════════════════════════════════════════
CDR      0.0  0.5  1.0  2.0  Total
cluster                           
0         44   25    8    1     78
1         56   16   11    0     83
2         35   29    9    1     74
Total    135   70   28    2    235

Row % (CDR proportion within each cluster):
CDR       0.0   0.5   1.0  2.0
cluster                       
0        56.4  32.1  10.3  1.3
1        67.5  19.3  13.3  0.0
2        47.3  39.2  12.2  1.4

═══════════════════════════════════════════════════════
  k=4  —  235 patients with CDR
═══════════════════════════════════════════════════════
CDR      0.0  0.5  1.0  2.0  Total
cluster                           
0         41   17    9    1     68
1         39   14    5    0     58
2         20   10    5    0     35
3         35   29    9    1     74
Total    135   70   28    2    235

Row % (CDR proportion within each cluster):
CDR       

### Step 8 — Silhouette score (all 416 patients)

In [18]:
sil_k3 = silhouette_score(D, labels_k3, metric="precomputed")
sil_k4 = silhouette_score(D, labels_k4, metric="precomputed")

print(f"Silhouette score (k=3) : {sil_k3:.4f}")
print(f"Silhouette score (k=4) : {sil_k4:.4f}")
print("(range −1 to 1, higher = better separated clusters)")

Silhouette score (k=3) : 0.3272
Silhouette score (k=4) : 0.3014
(range −1 to 1, higher = better separated clusters)


### Step 9 — ARI and NMI (235 patients with CDR)

CDR is multiplied by 2 to convert to integer categories before computing ARI and NMI.

In [19]:
mask_cdr  = cdr_series.notnull().values
# CDR × 2 → integer categories: 0→0, 0.5→1, 1→2, 2→4
cdr_int   = (cdr_series.values[mask_cdr] * 2).astype(int)

ari_k3 = adjusted_rand_score(cdr_int, labels_k3[mask_cdr])
nmi_k3 = normalized_mutual_info_score(cdr_int, labels_k3[mask_cdr])

ari_k4 = adjusted_rand_score(cdr_int, labels_k4[mask_cdr])
nmi_k4 = normalized_mutual_info_score(cdr_int, labels_k4[mask_cdr])

print(f"k=3  ARI={ari_k3:.4f}  NMI={nmi_k3:.4f}")
print(f"k=4  ARI={ari_k4:.4f}  NMI={nmi_k4:.4f}")
print(f"\nCDR integer categories used: {sorted(set(cdr_int))}")
print(f"Evaluated on: {mask_cdr.sum()} patients with CDR")

k=3  ARI=0.0160  NMI=0.0216
k=4  ARI=0.0006  NMI=0.0156

CDR integer categories used: [np.int64(0), np.int64(1), np.int64(2), np.int64(4)]
Evaluated on: 235 patients with CDR


### Step 10 — Summary table: k=3 vs k=4

In [22]:
summary = pd.DataFrame([
    {"k": 3,
     "Silhouette": round(sil_k3, 4),
     "ARI":        round(ari_k3, 4),
     "NMI":        round(nmi_k3, 4),
     "Silhouette evaluated on": f"all {len(patient_ids)} patients",
     "ARI/NMI evaluated on":    f"{mask_cdr.sum()} patients with CDR"},
    {"k": 4,
     "Silhouette": round(sil_k4, 4),
     "ARI":        round(ari_k4, 4),
     "NMI":        round(nmi_k4, 4),
     "Silhouette evaluated on": f"all {len(patient_ids)} patients",
     "ARI/NMI evaluated on":    f"{mask_cdr.sum()} patients with CDR"},
])

print("Combined clinical + all-images-merged PCA — Generalised Gower (d1=euclidean)")
print("=" * 70)
print(summary.to_string(index=False))
print("\nHigher = better for all three metrics.")

Combined clinical + all-images-merged PCA — Generalised Gower (d1=euclidean)
 k  Silhouette    ARI    NMI Silhouette evaluated on  ARI/NMI evaluated on
 3      0.3272 0.0160 0.0216        all 416 patients 235 patients with CDR
 4      0.3014 0.0006 0.0156        all 416 patients 235 patients with CDR

Higher = better for all three metrics.


## Save results

In [23]:
for labels, k in [(labels_k3, 3), (labels_k4, 4)]:
    df_out = pd.DataFrame({"patient_id": patient_ids, "cluster": labels})
    df_out = df_out.merge(
        cdr_series.reset_index().rename(columns={"CDR": "CDR"}),
        on="patient_id", how="left"
    )
    out_path = os.path.join(NB_DIR, f"combined_clinical_all_merged_k{k}_results.csv")
    df_out.to_csv(out_path, index=False)
    print(f"Saved k={k}: {out_path}")

Saved k=3: C:\Users\aregk\OneDrive\Desktop\Thesis_downoading_the_data\notebooks\combined_clinical_all_merged_k3_results.csv
Saved k=4: C:\Users\aregk\OneDrive\Desktop\Thesis_downoading_the_data\notebooks\combined_clinical_all_merged_k4_results.csv


---

## Change 2 — Per-image systematic experiment (each image type + clinical)

Loops over all 5 PCA files combined individually with clinical variables.
For each image type the matrix has **115 quantitative + 1 binary + 2 multiclass = 118 columns**.

| Group | Columns | p |
|-------|---------|---|
| Quantitative (`p1=115`) | 110 PCA + Age, MMSE, eTIV, nWBV, ASF | 115 |
| Binary (`p2=1`) | M/F | 1 |
| Multiclass (`p3=2`) | Educ, SES | 2 |

**Settings:** `metric='ggower'`, `d1='euclidean'`, `d2='sokal'`, `d3='hamming'`, `k=3` and `k=4`.

In [24]:
PCA_FILES = [
    "pca_cor.csv",
    "pca_gfc_sag.csv",
    "pca_gfc_tra.csv",
    "pca_masked_tra.csv",
    "pca_sbj_sag.csv",
]
K_VALUES      = [3, 4]
P1_IMG, P2_IMG, P3_IMG = 115, 1, 2
QUANT_CLIN    = ["Age", "MMSE", "eTIV", "nWBV", "ASF"]
KEEP_CLINICAL = ["patient_id", "Age", "M/F", "Educ", "SES", "MMSE", "eTIV", "nWBV", "ASF"]

per_image_results = []

for pca_file in PCA_FILES:
    image_type = pca_file.replace("pca_", "").replace(".csv", "")

    # ── Build combined feature matrix ────────────────────────────────────────
    df_img = pd.read_csv(os.path.join(NB_DIR, pca_file))
    df_mrg = df_img.merge(df_clin[KEEP_CLINICAL], on="patient_id", how="inner")

    # Fill missing
    for col in ["Educ", "SES", "MMSE"]:
        df_mrg[col] = df_mrg[col].fillna(df_mrg[col].median())

    # Encode M/F → binary
    df_mrg["MF"] = (df_mrg["M/F"] == "M").astype(int)
    df_mrg = df_mrg.drop(columns=["M/F"])
    df_mrg["Educ"] = df_mrg["Educ"].astype(int)
    df_mrg["SES"]  = df_mrg["SES"].astype(int)

    # Column order: PCA → quant clinical → binary → multiclass
    pca_cols_img  = [c for c in df_mrg.columns
                     if c not in ["patient_id", "Age", "MF", "Educ", "SES",
                                  "MMSE", "eTIV", "nWBV", "ASF"]]
    feat_cols_img = pca_cols_img + QUANT_CLIN + ["MF", "Educ", "SES"]
    pids_img      = df_mrg["patient_id"].values
    X_img         = df_mrg[feat_cols_img].to_numpy(dtype=np.float64)

    # CDR alignment
    cdr_img  = cdr_lookup.reindex(pids_img)
    mask_img = cdr_img.notnull().values
    cdr_int_img = (cdr_img.values[mask_img] * 2).astype(int)

    print(f"\n{'='*62}")
    print(f"  {image_type} + clinical  |  X={X_img.shape}  |  CDR: {mask_img.sum()}")
    print(f"{'='*62}")

    # ── Precompute distance matrix once per image (reused for k=3 and k=4) ──
    print("  Computing G-Gower distance matrix ...", end=" ", flush=True)
    D_img = generalized_gower_dist_matrix(
        X_img, p1=P1_IMG, p2=P2_IMG, p3=P3_IMG,
        d1="euclidean", d2="sokal", d3="hamming",
    )
    print("done")

    for k in K_VALUES:
        base = kmedoids.KMedoids(
            n_clusters=k, metric="precomputed", method="pam", random_state=42,
        )
        model = SampleDistClustering(
            clustering_method=base,
            metric="ggower",
            frac_sample_size=1.0,
            random_state=42,
            p1=P1_IMG, p2=P2_IMG, p3=P3_IMG,
            d1="euclidean", d2="sokal", d3="hamming",
        )
        print(f"  k={k} fitting ...", end=" ", flush=True)
        model.fit(X_img)
        labels_img = np.array(model.labels_, dtype=int)
        print("done")

        # % healthy per cluster
        df_eval = pd.DataFrame({"cluster": labels_img, "CDR": cdr_img.values})
        df_eval = df_eval[df_eval["CDR"].notnull()].copy()
        pct_tab = pd.crosstab(
            df_eval["cluster"], df_eval["CDR"], normalize="index"
        ).mul(100).round(1)
        pct_healthy = pct_tab[0.0] if 0.0 in pct_tab.columns \
                      else pd.Series(0.0, index=pct_tab.index)
        spread = float(pct_healthy.max() - pct_healthy.min())

        sil = silhouette_score(D_img, labels_img, metric="precomputed")
        ari = adjusted_rand_score(cdr_int_img, labels_img[mask_img])
        nmi = normalized_mutual_info_score(cdr_int_img, labels_img[mask_img])

        sizes = {c: int((labels_img == c).sum()) for c in sorted(set(labels_img))}
        print(f"    sizes={sizes}  spread={spread:.1f}%  sil={sil:.4f}  ARI={ari:.4f}  NMI={nmi:.4f}")

        per_image_results.append({
            "image_type":         image_type,
            "metric":             "ggower",
            "k":                  k,
            "silhouette":         round(sil, 4),
            "ari":                round(ari, 4),
            "nmi":                round(nmi, 4),
            "spread_pct_healthy": round(spread, 2),
        })

print("\n\nAll per-image experiments complete.")


  cor + clinical  |  X=(416, 118)  |  CDR: 235
  Computing G-Gower distance matrix ... done
  k=3 fitting ... done
    sizes={np.int64(0): 78, np.int64(1): 183, np.int64(2): 155}  spread=20.2%  sil=0.3284  ARI=0.0160  NMI=0.0216
  k=4 fitting ... done
    sizes={np.int64(0): 68, np.int64(1): 99, np.int64(2): 94, np.int64(3): 155}  spread=19.9%  sil=0.3034  ARI=0.0006  NMI=0.0156

  gfc_sag + clinical  |  X=(416, 118)  |  CDR: 235
  Computing G-Gower distance matrix ... done
  k=3 fitting ... done
    sizes={np.int64(0): 78, np.int64(1): 183, np.int64(2): 155}  spread=20.2%  sil=0.3284  ARI=0.0160  NMI=0.0216
  k=4 fitting ... done
    sizes={np.int64(0): 68, np.int64(1): 99, np.int64(2): 94, np.int64(3): 155}  spread=19.9%  sil=0.3034  ARI=0.0006  NMI=0.0156

  gfc_tra + clinical  |  X=(416, 118)  |  CDR: 235
  Computing G-Gower distance matrix ... done
  k=3 fitting ... done
    sizes={np.int64(0): 78, np.int64(1): 183, np.int64(2): 155}  spread=20.2%  sil=0.3284  ARI=0.0160  NMI=0.0

## Combined results — all experiments → export

In [25]:
def _spread(labels, cdr_s):
    df_e = pd.DataFrame({"cluster": labels, "CDR": cdr_s.values})
    df_e = df_e[df_e["CDR"].notnull()].copy()
    pct  = pd.crosstab(df_e["cluster"], df_e["CDR"], normalize="index").mul(100)
    ph   = pct[0.0] if 0.0 in pct.columns else pd.Series(0.0, index=pct.index)
    return round(float(ph.max() - ph.min()), 2)

# ── All-images-merged euclidean rerun (Exp 1, k=3 and k=4) ──────────────────
merged_rows = [
    {"image_type": "all_merged", "metric": "ggower", "k": 3,
     "silhouette": round(sil_k3, 4), "ari": round(ari_k3, 4), "nmi": round(nmi_k3, 4),
     "spread_pct_healthy": _spread(labels_k3, cdr_series)},
    {"image_type": "all_merged", "metric": "ggower", "k": 4,
     "silhouette": round(sil_k4, 4), "ari": round(ari_k4, 4), "nmi": round(nmi_k4, 4),
     "spread_pct_healthy": _spread(labels_k4, cdr_series)},
]

df_results = pd.DataFrame(merged_rows + per_image_results)

NB_PADDED = os.path.join(NB_DIR, "new_results_with_padding")
out_path  = os.path.join(NB_PADDED, "results_combined_clinical_image.csv")
df_results.to_csv(out_path, index=False)

print(f"Total rows : {len(df_results)}  (2 all_merged + {len(per_image_results)} per-image)")
print(f"\n{df_results.to_string(index=False)}")
print(f"\nSaved: {out_path}")

Total rows : 12  (2 all_merged + 10 per-image)

image_type metric  k  silhouette    ari    nmi  spread_pct_healthy
all_merged ggower  3      0.3272 0.0160 0.0216               20.17
all_merged ggower  4      0.3014 0.0006 0.0156               19.94
       cor ggower  3      0.3284 0.0160 0.0216               20.20
       cor ggower  4      0.3034 0.0006 0.0156               19.90
   gfc_sag ggower  3      0.3284 0.0160 0.0216               20.20
   gfc_sag ggower  4      0.3034 0.0006 0.0156               19.90
   gfc_tra ggower  3      0.3284 0.0160 0.0216               20.20
   gfc_tra ggower  4      0.3034 0.0006 0.0156               19.90
masked_tra ggower  3      0.3285 0.0160 0.0216               20.20
masked_tra ggower  4      0.3035 0.0006 0.0156               19.90
   sbj_sag ggower  3      0.3280 0.0160 0.0216               20.20
   sbj_sag ggower  4      0.3028 0.0006 0.0156               19.90

Saved: C:\Users\aregk\OneDrive\Desktop\Thesis_downoading_the_data\notebooks\new_

---

## Clinical only — G-Gower (baseline with no imaging features)

Uses only the 8 clinical variables — no PCA components.
This is a useful baseline: if clustering with clinical data alone already separates CDR groups,
the imaging features are not adding discriminative power.

| Group | Columns | p |
|-------|---------|---|
| Quantitative (`p1=5`) | Age, MMSE, eTIV, nWBV, ASF | 5 |
| Binary (`p2=1`) | M/F | 1 |
| Multiclass (`p3=2`) | Educ, SES | 2 |

`metric='ggower'`, `d1='euclidean'`, `d2='sokal'`, `d3='hamming'`, k=3 and k=4.

In [26]:
# df is already preprocessed (missing filled, M/F encoded, types set) — reuse it
CLIN_FEAT   = ["Age", "MMSE", "eTIV", "nWBV", "ASF", "MF", "Educ", "SES"]
P1_C, P2_C, P3_C = 5, 1, 2
X_clin = df[CLIN_FEAT].to_numpy(dtype=np.float64)
# patient_ids and cdr_series are already aligned to the same 416 patients

print(f"Clinical-only matrix : {X_clin.shape}  (p1={P1_C}, p2={P2_C}, p3={P3_C})")
print(f"Columns              : {CLIN_FEAT}")

# ── Precompute distance matrix once (reused for k=3 and k=4) ─────────────────
print("\nComputing G-Gower distance matrix ...", end=" ", flush=True)
D_clin = generalized_gower_dist_matrix(
    X_clin, p1=P1_C, p2=P2_C, p3=P3_C,
    d1="euclidean", d2="sokal", d3="hamming",
)
print(f"done  shape={D_clin.shape}  range=[{D_clin.min():.4f}, {D_clin.max():.4f}]")

clin_results = []

for k in [3, 4]:
    base = kmedoids.KMedoids(
        n_clusters=k, metric="precomputed", method="pam", random_state=42,
    )
    model = SampleDistClustering(
        clustering_method=base,
        metric="ggower",
        frac_sample_size=1.0,
        random_state=42,
        p1=P1_C, p2=P2_C, p3=P3_C,
        d1="euclidean", d2="sokal", d3="hamming",
    )
    print(f"\n  k={k} fitting ...", end=" ", flush=True)
    model.fit(X_clin)
    labels_c = np.array(model.labels_, dtype=int)
    print("done")

    # Crosstab
    df_ev = pd.DataFrame({"cluster": labels_c, "CDR": cdr_series.values})
    df_ev = df_ev[df_ev["CDR"].notnull()].copy()
    pct_t = pd.crosstab(df_ev["cluster"], df_ev["CDR"], normalize="index").mul(100).round(1)
    pct_h = pct_t[0.0] if 0.0 in pct_t.columns else pd.Series(0.0, index=pct_t.index)
    spread_c = float(pct_h.max() - pct_h.min())

    print(f"\n  Row % CDR (k={k}):")
    print(pct_t.to_string())

    sil_c = silhouette_score(D_clin, labels_c, metric="precomputed")
    ari_c = adjusted_rand_score(cdr_int, labels_c[mask_cdr])
    nmi_c = normalized_mutual_info_score(cdr_int, labels_c[mask_cdr])

    sizes_c = {c: int((labels_c == c).sum()) for c in sorted(set(labels_c))}
    print(f"\n  sizes={sizes_c}  spread={spread_c:.1f}%  sil={sil_c:.4f}  ARI={ari_c:.4f}  NMI={nmi_c:.4f}")

    clin_results.append({
        "image_type":         "clinical_only",
        "metric":             "ggower",
        "k":                  k,
        "silhouette":         round(sil_c, 4),
        "ari":                round(ari_c, 4),
        "nmi":                round(nmi_c, 4),
        "spread_pct_healthy": round(spread_c, 2),
    })

print("\nClinical-only experiments complete.")

Clinical-only matrix : (416, 8)  (p1=5, p2=1, p3=2)
Columns              : ['Age', 'MMSE', 'eTIV', 'nWBV', 'ASF', 'MF', 'Educ', 'SES']

Computing G-Gower distance matrix ... done  shape=(416, 416)  range=[0.0000, 5.9899]

  k=3 fitting ... done

  Row % CDR (k=3):
CDR       0.0   0.5   1.0  2.0
cluster                       
0        56.4  32.1  10.3  1.3
1        67.5  19.3  13.3  0.0
2        47.3  39.2  12.2  1.4

  sizes={np.int64(0): 78, np.int64(1): 183, np.int64(2): 155}  spread=20.2%  sil=0.3286  ARI=0.0160  NMI=0.0216

  k=4 fitting ... done

  Row % CDR (k=4):
CDR       0.0   0.5   1.0  2.0
cluster                       
0        60.3  25.0  13.2  1.5
1        67.2  24.1   8.6  0.0
2        57.1  28.6  14.3  0.0
3        47.3  39.2  12.2  1.4

  sizes={np.int64(0): 68, np.int64(1): 99, np.int64(2): 94, np.int64(3): 155}  spread=19.9%  sil=0.3039  ARI=0.0006  NMI=0.0156

Clinical-only experiments complete.


### Update results CSV — append clinical-only rows

In [27]:
df_results_full = pd.concat(
    [df_results, pd.DataFrame(clin_results)],
    ignore_index=True
)

NB_PADDED = os.path.join(NB_DIR, "new_results_with_padding")
out_path  = os.path.join(NB_PADDED, "results_combined_clinical_image.csv")
df_results_full.to_csv(out_path, index=False)

print(f"Total rows : {len(df_results_full)}")
print(f"\n{df_results_full.to_string(index=False)}")
print(f"\nSaved: {out_path}")

Total rows : 14

   image_type metric  k  silhouette    ari    nmi  spread_pct_healthy
   all_merged ggower  3      0.3272 0.0160 0.0216               20.17
   all_merged ggower  4      0.3014 0.0006 0.0156               19.94
          cor ggower  3      0.3284 0.0160 0.0216               20.20
          cor ggower  4      0.3034 0.0006 0.0156               19.90
      gfc_sag ggower  3      0.3284 0.0160 0.0216               20.20
      gfc_sag ggower  4      0.3034 0.0006 0.0156               19.90
      gfc_tra ggower  3      0.3284 0.0160 0.0216               20.20
      gfc_tra ggower  4      0.3034 0.0006 0.0156               19.90
   masked_tra ggower  3      0.3285 0.0160 0.0216               20.20
   masked_tra ggower  4      0.3035 0.0006 0.0156               19.90
      sbj_sag ggower  3      0.3280 0.0160 0.0216               20.20
      sbj_sag ggower  4      0.3028 0.0006 0.0156               19.90
clinical_only ggower  3      0.3286 0.0160 0.0216               20.20
cli